In [1]:
import arcpy
from arcpy import env
import os
import numpy as np
from arcgis import GIS
from arcgis.features import GeoAccessor
from arcgis.features import GeoSeriesAccessor
import pandas as pd

arcpy.env.overwriteOutput = True
arcpy.env.parallelProcessingFactor = "90%"

# show all columns
pd.options.display.max_columns = None

# pd.pivot_table(df, values='a', index='b', columns='c', aggfunc='sum', fill_value=0)
# pd.DataFrame.spatial.from_featureclass(???)  
# df.spatial.to_featureclass(location=???,sanitize_columns=False)  

# gsa = arcgis.features.GeoSeriesAccessor(df['SHAPE'])  
# df['AREA'] = gsa.area  # KNOW YOUR UNITS

In [2]:
## spatial join
# target_features = ?
# join_features = ?
# output_features = os.path.join(gdb, ?)

# fieldmappings = arcpy.FieldMappings()
# fieldmappings.addTable(target_features)
# fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

# sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
# sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [3]:
# fill NA values in Spatially enabled dataframes (ignores SHAPE column)
def fill_na_sedf(df_with_shape_column, fill_value=0):
    if 'SHAPE' in list(df_with_shape_column.columns):
        cols_to_fill = df_with_shape_column.columns.difference(['SHAPE'])
        df_with_shape_column[cols_to_fill] = df_with_shape_column[cols_to_fill].fillna(fill_value)
        return df_with_shape_column
    else:
        raise Exception("Dataframe does not include 'SHAPE' column")

In [4]:
outputs = ['.\\Outputs', "scratch.gdb", 'results.gdb']

if not os.path.exists(outputs[0]):
    os.makedirs(outputs[0])

gdb = os.path.join(outputs[0], outputs[1])
gdb2 = os.path.join(outputs[0], outputs[2])

if not arcpy.Exists(gdb):
    arcpy.CreateFileGDB_management(outputs[0], outputs[1])

if not arcpy.Exists(gdb2):
    arcpy.CreateFileGDB_management(outputs[0], outputs[2])

In [7]:
parcels = r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels'
new_mag_pts = r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Default.gdb\mag_pts_20260128'

In [27]:
# spatial join
target_features = parcels
join_features = new_mag_pts
output_features = os.path.join(gdb, "parcels_mag_pts_sj")

fieldmappings = arcpy.FieldMappings()
fieldmappings.addTable(target_features)
fieldmappings.addTable(join_features)

# # variable
# fieldindex = fieldmappings.findFieldMapIndex(?)
# fieldmap = fieldmappings.getFieldMap(fieldindex)
# fieldmap.mergeRule = 'Sum'
# fieldmappings.replaceFieldMap(fieldindex, fieldmap)

sj = arcpy.SpatialJoin_analysis(target_features, join_features, output_features,'JOIN_ONE_TO_ONE', "KEEP_ALL", fieldmappings, match_option="INTERSECT")
sj_df = pd.DataFrame.spatial.from_featureclass(sj[0]).copy()

In [14]:
sj_df.head()

,OBJECTID,Join_Count,TARGET_FID,parcel_id,WFRC_parcel_id,county_id,CO_NAME,year_built,total_market_value,land_value,building_id,building_type_id,building_type,building_sqft,non_residential_sqft,residential_units,job_spaces,stories,unit_price_non_residential,res_price_per_sqft,basebldg,redev_friction,NoBuild,IS_OUG,parcel_acres,Tax_Exempt,parent_parcel,volume_one_way,volume_two_way,volume_two_way_nofwy,zonal_ppa,x,y,note,parcel_sqft,Split,Split_Factor,MAG_parcel_id,max_far,max_dua,type1,type2,type3,type4,type5,type6,type7,type8,TAZID_900,distsml_id,distmed_id,distlrg_id,CITY_NAME,stream_dist,streams,trail_dist,trail,airport_distance,airport,fwy_exit_dist,fwy_exit_new,bus_stop_dist_new,bus_stop_new,bus_rte_dist,rail_stn_dist,rail_stn_new,raildepot_dist,rail_depot,university_dist,university,elevation,agriculture,TAZID_910,grid_id,MKT_CNTVAL,NEW_YEAR,SHAPE
0,1,0,1,0,0,11,DAVIS,-9999,7344,7344,0,-9999,,-9999,-9999,0,0,0,-9999,-9999.0,-9999,-9999,1,0,5.217058,1,453823,0,2,2,122240.3443,422969.78497,4534584.510975,base,2046625.968711,1,,,0.5,0.2,1,0,0,0,0,0,0,0,791,30,22,9,Farmington,365.052708,0,878.627247,0,9976.158841,0,1902.254794,0,2611.874229,0,2020.163073,3312.735097,0,21687.661437,0,2962.804147,0,1284,0,791,82440,<NA>,<NA>,"{""rings"": [[[422916.5438000001, 4534528.854800..."
1,2,0,2,1,1,11,DAVIS,-9999,7344,7344,1,-9999,,-9999,-9999,0,0,0,-9999,-9999.0,-9999,-9999,1,0,5.217057,1,453823,0,2,2,122240.3443,422968.46205,4534687.611511,base,2046625.968711,1,,,0.5,0.2,1,0,0,0,0,0,0,0,791,30,22,9,Farmington,411.962576,0,799.515279,0,10078.524868,0,1914.063596,0,2510.469878,0,2022.800505,3215.141895,0,21790.803729,0,2863.706391,0,1284,0,791,82440,<NA>,<NA>,"{""rings"": [[[422868.51219999976, 4534739.63759..."
2,3,0,3,2,2,11,DAVIS,-9999,7344,7344,2,-9999,,-9999,-9999,0,0,0,-9999,-9999.0,-9999,-9999,1,0,5.217056,1,453823,0,2,2,122240.3443,422970.277028,4534792.291682,base,2046625.968711,1,,,0.5,0.2,1,0,0,0,0,0,0,0,791,30,22,9,Farmington,392.481566,0,720.237927,0,10182.81665,0,1928.515649,0,2407.090481,0,2022.319521,3115.388244,0,21895.477121,0,2762.510272,0,1284,0,791,82637,<NA>,<NA>,"{""rings"": [[[422868.51219999976, 4534739.63759..."
3,4,0,4,3,3,11,DAVIS,-9999,7344,7344,3,-9999,,-9999,-9999,0,0,0,-9999,-9999.0,-9999,-9999,1,0,5.217059,1,453823,0,2,2,122240.3443,422972.140373,4534898.908811,base,2046625.968711,1,,,0.5,0.2,1,0,0,0,0,0,0,0,791,30,22,9,Farmington,359.17827,0,643.35752,0,10289.0489,0,1948.90386,0,2301.920984,0,2021.900958,3014.130903,0,22002.087615,0,2659.718688,0,1284,0,791,82637,<NA>,<NA>,"{""rings"": [[[422872.17509999964, 4534845.2684]..."
4,5,0,5,4,4,11,DAVIS,-9999,7344,7344,4,-9999,,-9999,-9999,0,0,0,-9999,-9999.0,-9999,-9999,1,0,5.217057,1,453823,0,2,2,122240.3443,423132.255553,4534616.102256,base,2046625.968711,1,,,0.5,0.2,1,0,0,0,0,0,0,0,791,30,22,9,Farmington,528.242627,0,739.640989,0,10026.751354,0,1743.104263,0,2527.992319,0,1858.055191,3236.075956,0,21717.32129,0,2893.81689,0,1284,0,791,82441,<NA>,<NA>,"{""rings"": [[[423070.216, 4534701.0679], [42319..."


In [28]:
sj_df.loc[
    ((sj_df['year_built'].isna()) | (sj_df['year_built'] == -9999) | (sj_df['NEW_YEAR'] > sj_df['year_built'])) & 
    (sj_df['county_id'] == 49) & 
    (sj_df['NEW_YEAR'].notna()) & 
    (sj_df['NEW_YEAR'] > 0)
,'year_built'] = sj_df['NEW_YEAR']

del sj_df['NEW_YEAR']
del sj_df['MKT_CNTVAL']

In [29]:
sj_df.spatial.to_featureclass(location=r'E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\remm_base_year_data.gdb\parcels_NEW',sanitize_columns=False)
sj_df.drop(['SHAPE', 'OBJECTID'], axis=1).to_csv(r"E:\Tasks\REMM-Manage-Base-Year-Data-2023\Inputs\Tables\parcels_20260129.csv",  index=False)  